<a href="https://colab.research.google.com/github/alexaK88/Fun_tasks_for_the_weekend/blob/main/word2vec_numpy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [43]:
import re
import numpy as np
from tqdm import tqdm
from collections import Counter

In [44]:
text_path = "/content/drive/MyDrive/star_rover.txt"

Here we will focus on one of my favourite books - The Jacket (aka, Star-Rover) by Jack London.


# Word2Vec rebuilt

Pipeline:
1. Load and tokenize the text
2. Build vocabulary
3. Encode corpus
4. Generate training pairs
5. Implement negative sampler
6. Initialize embeddings
7. Implement sigmoid
8. Implement forward pass
9. Implement loss
10. Implement gradients
11. Implement SGD update
12. Train

First, word2vec method is an NLP technique that transforms words into numerical vectors that capture semantic relationships. It maps words with similar meanings to close vector positions in a multi-dimensional space, enabling semantic calculations.
It's designed to generate meaningful word embeddings.

It offers two methods, skip-grams and CBOW, and I used both here to analyse the training.

Core problem for word2vec: given two words, determine whether they are neighbours. By neighbours we mean here if the words are within certain range of words away from eahc other or not.

Dataset for this problem is constructed by pairing the target word with each of its context words (neighbouring words of the target) as inputs and the output will be a label showing whether the target and context words are neighbours (1=yes, 0=no).


Problem is the following: given a target word and a context word, predict the probability that they are neighbours.

### Skip-grams



In [45]:
def load_text(path):
  with open(path, "r", encoding="utf-8") as f:
    text = f.read().lower() # lowercase the whole text
  return text

def tokenize(text):
    # tokens = re.findall(r"\b[a-z]+\b", text) # we will only keep words, no punctuation or anything else
    tokens = re.findall(r"[A-Za-z]+[\w^\']*|[\w^\']*[A-Za-z]+[\w^\']*", text)
    return tokens

def build_vocabulary(tokens, min_count=5):
    counts = Counter(tokens)
    vocabulary = [w for w, c in counts.items() if c >= min_count]

    word2ix = {w:i for i, w in enumerate(vocabulary)}
    idx2word = {i:w for w, i in word2ix.items()}

    word_freq = np.array([counts[w] for w in vocabulary])

    return word2ix, idx2word, word_freq

Negative sampling introduces pairs of words that are not neighbours. It's used to show the model which non-neighbours look like, i.e. this is how we add samples with labels 0 to the dataset.

Negative sampling steps:
1. Compute word frequencies
2. Apply Mikolov distribution
3. Randomly sample K negative words

In [46]:
class NegativeSampler:
    def __init__(self, word_freq, power=0.75):
        freq = np.array(word_freq) ** power
        self.prob = freq / freq.sum()

    def sample(self, K, forbidden=None):
        samples = []

        while len(samples) < K:
            w = np.random.choice(len(self.prob), p=self.prob)
            if forbidden is None or w not in forbidden:
                samples.append(w)

        return np.array(samples)

def subsample_tokens(encoded, word_freq, t=5e-5):
    total = np.sum(word_freq)
    freq = np.array(word_freq) / total

    keep_probs = np.sqrt(t / freq) + (t / freq)

    keep_probs = np.minimum(1.0, keep_probs)

    mask = np.random.rand(len(encoded)) < keep_probs[encoded]

    return encoded[mask]

In [47]:
def generate_skipgram_pairs(encoded, window_size):

    for i, center in enumerate(encoded):

        start = max(0, i - window_size)
        end = min(len(encoded), i + window_size + 1)

        for j in range(start, end):

            if i == j:
                continue

            context = encoded[j]

            yield center, context


def sigmoid(x):
    x = np.clip(x, -10, 10)
    return 1 / (1 + np.exp(-x))

In [48]:
class Word2VecSGNS:

    def __init__(self, vocab_size, embed_dim=100, lr=0.025):

        self.V = vocab_size
        self.D = embed_dim
        self.lr = lr

        # input embeddings
        self.W_in = np.random.randn(self.V, self.D) * 0.01

        # output embeddings
        self.W_out = np.random.randn(self.V, self.D) * 0.01

    def train_pair(self, center, context, negatives):
        """
        One SGD update step.
        """

        v_c = self.W_in[center]
        v_o = self.W_out[context]

        neg_vectors = self.W_out[negatives]

        # ---------- Forward ----------
        pos_score = np.dot(v_o, v_c)
        neg_scores = neg_vectors @ v_c

        pos_sig = sigmoid(pos_score)
        neg_sig = sigmoid(neg_scores)

        # Loss (optional for logging)
        loss = -np.log(pos_sig + 1e-10) \
               -np.sum(np.log(1 - neg_sig + 1e-10))

        # ---------- Gradients ----------

        # positive gradient
        grad_pos = pos_sig - 1

        # negative gradients
        grad_neg = neg_sig

        # gradient wrt center embedding
        grad_center = (
            grad_pos * v_o +
            np.sum(grad_neg[:, None] * neg_vectors, axis=0)
        )

        # ---------- Updates ----------

        # update output embeddings
        self.W_out[context] -= self.lr * grad_pos * v_c

        self.W_out[negatives] -= (
            self.lr * grad_neg[:, None] * v_c
        )

        # update center embedding
        self.W_in[center] -= self.lr * grad_center

        return loss


In [49]:
# creating list of tokens
text = load_text(text_path)
tokens = tokenize(text)

# building vocabulary - all words/tokens are unique
word2idx, idx2word, word_freq = build_vocabulary(tokens)

# generating training pairs; we are creating pairs like (center_word, context_word)
encoded = np.array([word2idx[w] for w in tokens if w in word2idx],
                   dtype=np.int32)

In [50]:
word2idx

{'the': 0,
 'project': 1,
 'gutenberg': 2,
 'ebook': 3,
 'of': 4,
 'jacket': 5,
 'star': 6,
 'rover': 7,
 'this': 8,
 'is': 9,
 'for': 10,
 'use': 11,
 'anyone': 12,
 'anywhere': 13,
 'in': 14,
 'united': 15,
 'states': 16,
 'and': 17,
 'most': 18,
 'other': 19,
 'world': 20,
 'at': 21,
 'no': 22,
 'with': 23,
 'almost': 24,
 'you': 25,
 'may': 26,
 'copy': 27,
 'it': 28,
 'give': 29,
 'away': 30,
 'or': 31,
 're': 32,
 'under': 33,
 'terms': 34,
 'license': 35,
 'www': 36,
 'org': 37,
 'if': 38,
 'are': 39,
 'not': 40,
 'located': 41,
 'will': 42,
 'have': 43,
 'to': 44,
 'laws': 45,
 'country': 46,
 'where': 47,
 'before': 48,
 'using': 49,
 'jack': 50,
 'date': 51,
 'language': 52,
 'english': 53,
 'david': 54,
 'price': 55,
 'start': 56,
 'by': 57,
 'chapter': 58,
 'i': 59,
 'all': 60,
 'my': 61,
 'life': 62,
 'had': 63,
 'an': 64,
 'awareness': 65,
 'times': 66,
 'places': 67,
 'been': 68,
 'aware': 69,
 'me': 70,
 'oh': 71,
 'trust': 72,
 'so': 73,
 'reader': 74,
 'that': 75,
 'b

In [51]:
WINDOW = 3
NEG_SAMPLES = 7
EPOCHS = 100

encoded = subsample_tokens(encoded, word_freq)
sampler = NegativeSampler(word_freq)

In [52]:
model = Word2VecSGNS(len(word2idx), embed_dim=100)


initial_lr = 0.05
total_steps = EPOCHS * len(encoded)
step = 0

for epoch in range(EPOCHS):
    total_loss = 0

    for center, context in tqdm(
        generate_skipgram_pairs(encoded, WINDOW)
    ):
        progress = step / total_steps
        model.lr = initial_lr * (1 - progress)
        model.lr = max(model.lr, initial_lr * 0.0001)
        negatives = sampler.sample(NEG_SAMPLES, forbidden={center, context})

        loss = model.train_pair(center, context, negatives)
        total_loss += loss

    print(f"Epoch {epoch+1} loss: {total_loss:.2f}")


173556it [01:33, 1853.73it/s]


Epoch 1 loss: 540477.39


173556it [01:27, 1974.76it/s]


Epoch 2 loss: 479812.42


173556it [01:28, 1950.26it/s]


...

Epoch 98 loss: 249482.65


173556it [01:27, 1989.93it/s]


Epoch 99 loss: 250010.58


173556it [01:27, 1972.71it/s]

Epoch 100 loss: 249636.30


### CBOW

In [53]:
class Word2VecCBOW:

    def __init__(self, vocab_size, embed_dim=100, lr=0.025):

        self.V = vocab_size
        self.D = embed_dim
        self.lr = lr

        self.W_in = np.random.randn(self.V, self.D) * 0.01
        self.W_out = np.random.randn(self.V, self.D) * 0.01

    def train_example(self, context, target, negatives):

        # ---------- Forward ----------

        context_vectors = self.W_in[context]
        v_context = np.mean(context_vectors, axis=0)

        v_target = self.W_out[target]
        neg_vectors = self.W_out[negatives]

        pos_score = np.dot(v_target, v_context)
        neg_scores = neg_vectors @ v_context

        pos_sig = sigmoid(pos_score)
        neg_sig = sigmoid(neg_scores)

        loss = -np.log(pos_sig + 1e-10) \
               - np.sum(np.log(1 - neg_sig + 1e-10))

        # ---------- Gradients ----------

        grad_pos = pos_sig - 1
        grad_neg = neg_sig

        # gradient wrt averaged context vector
        grad_context = (
            grad_pos * v_target +
            np.sum(grad_neg[:, None] * neg_vectors, axis=0)
        )

        # ---------- Updates ----------

        # update output embeddings
        self.W_out[target] -= self.lr * grad_pos * v_context

        self.W_out[negatives] -= (
            self.lr * grad_neg[:, None] * v_context
        )

        # distribute gradient equally to context words
        grad_each = grad_context / len(context)

        self.W_in[context] -= self.lr * grad_each

        return loss


In [54]:
def cbow_generator(encoded, max_window=5):

    for i in range(len(encoded)):

        start = max(0, i - max_window)
        end = min(len(encoded), i + max_window + 1)

        context = [
            encoded[j]
            for j in range(start, end)
            if j != i
        ]

        if len(context) == 0:
            continue

        center = encoded[i]

        yield context, center


In [55]:
model = Word2VecCBOW(len(word2idx), embed_dim=100)
sampler = NegativeSampler(word_freq)

total_steps = EPOCHS * len(encoded)
step = 0

# training loop with CBOW
for epoch in range(EPOCHS):

    total_loss = 0

    for context, center in tqdm(
        cbow_generator(encoded, WINDOW)
    ):
        progress = step / total_steps
        model.lr = initial_lr * (1 - progress)
        model.lr = max(model.lr, initial_lr * 0.0001)

        negatives = sampler.sample(NEG_SAMPLES)

        loss = model.train_example(context, center, negatives)
        total_loss += loss

    print(f"Epoch {epoch+1} loss: {total_loss:.2f}")


28928it [00:17, 1671.15it/s]


Epoch 1 loss: 144489.58


28928it [00:17, 1697.48it/s]


Epoch 2 loss: 97118.44


28928it [00:16, 1751.04it/s]


...

Epoch 98 loss: 6621.59


28928it [00:17, 1701.22it/s]


Epoch 99 loss: 6520.50


28928it [00:17, 1642.34it/s]

Epoch 100 loss: 6480.40


So, CBOW works much better than skip-gram with negative sampling.
Main reason: skip-grams create more training examples.
For instance, if we have window size = 2, then we will have 4 samples for this word (word + neighbour). Skip-gram loss accumulates many more terms.

CBOW converges faster, but it doesn't necessarily mean that the model is better than skip-grams. It's faster, cause, in comparison,
- skip-gram complexity: O(window x K)
- CBOW complexity: O(K)

But we can check in other way...

In [ ]:
def most_similar(word, k=5):
    idx = word2idx[word]
    vec = model.W_in[idx]

    norms = np.linalg.norm(model.W_in, axis=1)
    sims = model.W_in @ vec / (norms * np.linalg.norm(vec) + 1e-9)

    best = np.argsort(-sims)[1:k+1]
    return [idx2word[i] for i in best]


In [58]:
most_similar("king")

['heartily', 'raa', 'roof', 'protected', 'beggars']